In [2]:
%load_ext gradio

In [1]:
import gradio as gr

import os

In [ ]:
# CARPETA_AUDIOS = "./F5TTS/MarcosGrabacion523_Intento1"
# CARPETA_AUDIOS = "/mnt/SSD2T/2-DiplomadoIA/DeepLearning/dockersdata/speechAudiosContainer/F5TTS/MarcosGrabacion523_Intento1"
CARPETA_AUDIOS = "./F5TTS/NochesSanDamian"

In [ ]:
%%blocks

def greet(name):
    return "Hello " + name + "!"

with gr.Blocks() as demo:
    name = gr.Textbox(label="NOMBRES")
    output = gr.Textbox(label="Output Box")
    greet_btn = gr.Button("Greet")
    greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

# demo.launch(server_name="0.0.0.0", server_port=7860)

In [4]:
texto="hola"

def listar_audios():
    return [
        f for f in os.listdir(CARPETA_AUDIOS)
        if f.endswith(".wav")
    ]

def listar_rutas():
    return [os.path.join(CARPETA_AUDIOS, a) for a in listar_audios()]


def reproducir_audio(nombre_archivo):
    ruta_completa = os.path.join("/mnt/SSD2T/2-DiplomadoIA/DeepLearning/dockersdata/speechAudiosContainer/F5TTS/MarcosGrabacion523_Intento1/", nombre_archivo)
    print(ruta_completa)
    mensaje = f"🎵 Reproduciendo: **{nombre_archivo}**"
    return ruta_completa, mensaje


def welcome(name):
    return f"Welcome to Gradio, {name}!"



In [ ]:
%%blocks

with gr.Blocks() as demo:
    gr.Markdown("### Reproductor de Audios desde Volumen Docker")
    
    lista_archivos = gr.Dropdown(choices=listar_audios(), label="Selecciona un audio")

    # audio_output = gr.Audio(label="Reproducción", type="filepath")
    audio_output = gr.Audio(label="Reproducción", type="filepath")
  
    btn_reproducir = gr.Button("Reproducir")

    btn_reproducir.click(fn=reproducir_audio, inputs=lista_archivos, outputs=[audio_output,markdown_status])

    markdown_status=gr.Markdown("Hola")
    
    gr.Markdown(
    """
    # Hello World!
    Start typing below to see the output.
    """)
    inp = gr.Textbox(placeholder="What is your name?")
    out = gr.Textbox()
    inp.change(welcome, inp, out)

In [ ]:
%%blocks

with gr.Blocks() as demo:
    gr.Markdown("### Lista de Audios")

    archivos = listar_audios()

    for archivo in archivos:
        gr.Markdown(f"**{archivo}**")
        gr.Audio(
            value=os.path.join(CARPETA_AUDIOS, archivo),
            type="filepath",
            label=f"Reproductor de {archivo}"
        )

In [ ]:
%%blocks

with gr.Blocks() as demo:
    gr.Markdown("### Audios con volumen inicial")

    archivos = listar_audios()

    for i, archivo in enumerate(archivos):
        path = os.path.join(CARPETA_AUDIOS, archivo)

        gr.Markdown(f"**{archivo}**")
        audio_id = f"audio-{i}"

        # Reproductor de audio estándar
        gr.Audio(value=path, type="filepath", label=f"Audio {i}", elem_id=audio_id)

        # Inyección de JavaScript para ajustar el volumen
        gr.HTML(f"""
        <script>
        setTimeout(() => {{
            var player = document.querySelector("#{audio_id} audio");
            if (player) {{
                player.volume = 0.5;  // Volumen entre 0.0 y 1.0
            }}
        }}, 500);
        </script>
        """)

In [ ]:
%%blocks

with gr.Blocks() as demo:
    gr.Markdown("### Reproductor Personalizado con Teclado")

    archivos = listar_audios()
    rutas = listar_rutas()

    # Construye el HTML con elementos <audio> y un script JS
    html = """
    <div id="player-container">
    """

    for i, (ruta, nombre) in enumerate(zip(rutas, archivos)):
        html += f"""
        <p><strong>{nombre}</strong></p>
        <audio id="audio{i}" src="file/{ruta}" preload="auto"></audio>
        """

    html += """
    </div>
    <p>Pulsa la tecla <code>ESPACIO</code> para reproducir el siguiente audio.</p>
    <script>
        let audios = document.querySelectorAll("audio");
        let current = 0;

        function reproducirSiguiente() {
            if (current < audios.length) {
                let audio = audios[current];
                audio.volume = 0.7;
                audio.play();
                current++;
            }
        }

        document.addEventListener("keydown", function(e) {
            if (e.code === "Space") {
                e.preventDefault();
                reproducirSiguiente();
            }
        });
    </script>
    """

    gr.HTML(html)
